# Topic Modeling with LDA

Discover **latent themes** in review text without labels.

**Use case:** Explore themes in feedback, tickets, or survey responses.

**Prerequisites:** `01-nlp-fundamentals.ipynb` introduces the concepts. This notebook uses `nlp_helpers.py` for preprocessing.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

from nlp_helpers import DATASETS_DIR, download_nltk_data, preprocess_text

download_nltk_data()
print('Setup complete.')


## 1. Load documents

We reuse restaurant reviews as the corpus.


In [ ]:
path = f'{DATASETS_DIR}/restaurant-reviews.tsv'
df = pd.read_csv(path, sep='\t', quoting=3)
df.columns = ['Review', 'Liked']
df['processed'] = df['Review'].apply(preprocess_text)
print(len(df), 'documents')


## 2. Fit LDA

LDA assumes each document mixes topics; each topic mixes words. `CountVectorizer` supplies the word-document matrix.


In [ ]:
vec_lda = CountVectorizer(max_features=500, max_df=0.95, min_df=2)
X_lda = vec_lda.fit_transform(df['processed'])

n_topics = 3
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=20)
lda.fit(X_lda)

# Top words per topic
feature_names = vec_lda.get_feature_names_out()
n_top_words = 8

print("Discovered topics in restaurant reviews:\n")
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-n_top_words:][::-1]]
    print(f"Topic {topic_idx + 1}: " + ", ".join(top_words))